In [0]:
  import pyspark.sql.functions as F
  from pyspark.sql.types import *

In [0]:
bronze_catalog_name = 'ecommerce.bronze'

In [0]:
def write_silver_table(df, dim: str) -> None:
    df.write.mode('overwrite')\
        .format('delta')\
        .option('mergeSchema', 'true')\
        .saveAsTable(f'ecommerce.silver.silver_{dim}')

## Brands

In [0]:
df_bronze_brands = spark.table(f'{bronze_catalog_name}.bronze_brands')

In [0]:
df_bronze_brands.show(10)

In [0]:
df_silver_brands = df_bronze_brands\
    .withColumn('brand_name', F.trim(F.col('brand_name')))\
    .withColumn('brand_code', F.regexp_replace(F.col('brand_code'), '[^a-zA-Z0-9]', ''))

df_silver_brands.show(10)

In [0]:
df_silver_brands.select('category_code').distinct().show()

In [0]:
category_discrepancies = {
    'BOOKS': 'BKS',
    'GROCERY': 'GRCY',
    'TOYS': 'TOY'
}

df_silver_brands = df_silver_brands.replace(category_discrepancies, subset=['category_code'])
df_silver_brands.select('category_code').distinct().show()

In [0]:
write_silver_table(df_silver_brands, 'brands')

## Category

In [0]:
df_bronze_category = spark.table(f'{bronze_catalog_name}.bronze_category')

In [0]:
df_bronze_category.show(10)

In [0]:
df_bronze_category.groupBy('category_code').count().filter(F.col('count') > 1).show()

In [0]:
df_silver_category = df_bronze_category\
    .dropDuplicates(['category_code'])\
    .withColumn('category_code', F.upper(F.col('category_code')))

df_silver_category.show(10)

In [0]:
write_silver_table(df_silver_category, 'category')

## Products

In [0]:
df_bronze_products = spark.table(f'{bronze_catalog_name}.bronze_products')

In [0]:
display(df_bronze_products.limit(10))

In [0]:
df_silver_products = df_bronze_products\
    .withColumn('weight_grams', F.regexp_replace(F.col('weight_grams'), 'g', ''))\
    .withColumn('weight_grams', F.col('weight_grams').cast(IntegerType()))

df_silver_products.printSchema()

In [0]:
df_silver_products = df_silver_products\
    .withColumn('lenght_cm', F.regexp_replace(F.col('lenght_cm'), ',', '.'))\
    .withColumn('lenght_cm', F.col('lenght_cm').cast(FloatType()))

display(df_silver_products.select('lenght_cm').limit(1))

In [0]:
df_silver_products = df_silver_products\
    .withColumn('category_code', F.upper(F.col("category_code")))\
    .withColumn('brand_code', F.upper(F.col("brand_code")))

display(df_silver_products.select("brand_code",'category_code').limit(5))

In [0]:
df_silver_products.select('material').distinct().show()

In [0]:
fix_spelling = {
    'Alumium': 'Aluminum',
    'Coton': 'Cotton',
    'Ruber': 'Rubber'
}

df_silver_products = df_silver_products.replace(fix_spelling, subset=['material'])

df_silver_products.select('material').distinct().show()

In [0]:
df_silver_products.select('rating_count').filter(F.col('rating_count')<0).show(3)

In [0]:
df_silver_products = df_silver_products\
    .withColumn('rating_count', F.abs(F.col('rating_count')))\
    .withColumn(
        'rating_count', 
        F.when(F.col('rating_count').isNull(),0).otherwise(F.col('rating_count'))
    )

df_silver_products.select('rating_count').filter(F.col('rating_count').isNull()).show(3)


In [0]:
write_silver_table(df_silver_products, 'products')

## Customers

In [0]:
df_bronze_customers = spark.table(f'{bronze_catalog_name}.bronze_customers')

In [0]:
df_bronze_customers.filter(F.col('customer_id').isNull()).count()

In [0]:
df_silver_customers = df_bronze_customers.dropna(subset=['customer_id'])

df_silver_customers.filter(F.col('customer_id').isNull()).count()

In [0]:
df_silver_customers.filter(F.col('phone').isNull()).count()

In [0]:
df_silver_customers = df_silver_customers.fillna(subset=['phone'], value='Not available')

df_silver_customers.filter(F.col('phone').isNull()).count()

In [0]:
df_silver_customers.filter(F.col('phone')=='Not available').show(5)

In [0]:
write_silver_table(df_silver_customers, 'customers')

## Date

In [0]:
df_bronze_date = spark.table(f'{bronze_catalog_name}.bronze_date')

In [0]:
display(df_bronze_date.limit(10))

In [0]:
df_silver_date = df_bronze_date\
    .withColumn('date', F.to_date(F.col('date'), 'dd-MM-yyyy'))

df_silver_date.printSchema()

In [0]:
df_silver_date.groupBy("date").count().filter(F.col('count')>1).show(10)

In [0]:
df_silver_date = df_silver_date.dropDuplicates(["date"])
df_silver_date.groupBy("date").count().filter(F.col('count')>1).show(10)

In [0]:
df_silver_date = df_silver_date\
    .withColumn('day_name', F.initcap(F.col("day_name")))

df_silver_date.select("day_name").show(5)

In [0]:
df_silver_date = df_silver_date\
    .withColumn('week_of_year', F.abs(F.col('week_of_year')))

df_silver_date.select("week_of_year").show(3)

In [0]:
df_silver_date = df_silver_date\
    .withColumn('quarter', F.concat(F.lit('Q'), F.col('quarter'), F.lit('-'), F.col('year')))

df_silver_date.select("quarter").show(3)


In [0]:
df_silver_date = df_silver_date\
    .withColumn('week_of_year', F.concat(F.lit('Week'), F.col('week_of_year'), F.lit('-'), F.col('year')))

df_silver_date.select("week_of_year").show(3)

In [0]:
display(df_silver_date.limit(5))

In [0]:
write_silver_table(df_silver_date, 'date')